# 🤖 Smart Science Chatbot
### Ask any question — the bot finds the best answer using NLP + TF-IDF retrieval
---
**How it works:**
```
User types a question
       ↓
TF-IDF similarity → find closest matching question in dataset
       ↓
Naive Bayes intent classifier → understand what kind of question it is
       ↓
Return reference answer + explanation + confidence
       ↓
If no match found → fallback response
```

## 📦 Step 0 — Install & Import

In [ ]:
!pip install nltk scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import re
import io
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder

for r in ['stopwords','punkt','wordnet','punkt_tab']:
    nltk.download(r, quiet=True)

print('✅ Ready!')

## 📂 Step 1 — Load Dataset

In [ ]:
# ── Jupyter: place CSV in same folder as notebook ──
df = pd.read_csv('test-uq-00001.csv')

# ── Google Colab: uncomment below ──
# from google.colab import files
# uploaded = files.upload()
# for filename in uploaded:
#     df = pd.read_csv(io.StringIO(open(filename).read()))

# Build a clean knowledge base: one row per unique question
kb = (
    df.groupby('question')
      .agg(reference_answer=('reference_answer', 'first'))
      .reset_index()
)

print(f'✅ Knowledge base built: {len(kb)} unique Q&A pairs')
kb.head(3)

## 🧹 Step 2 — Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def preprocess(text: str) -> str:
    if not text or pd.isna(text):
        return ''
    text   = str(text).lower()
    text   = re.sub(r'[^a-z\s]', ' ', text)
    text   = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
               if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

kb['question_clean'] = kb['question'].apply(preprocess)
kb['answer_clean']   = kb['reference_answer'].apply(preprocess)

print('✅ Knowledge base preprocessed.')

## 🧠 Step 3 — Build TF-IDF Retrieval Engine
Converts every question into a vector. When user asks something, we find the closest match using **cosine similarity**.

In [ ]:
# Fit TF-IDF on the knowledge base questions
retrieval_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True
)

kb_vectors = retrieval_tfidf.fit_transform(kb['question_clean'])

print(f'✅ TF-IDF retrieval engine built.')
print(f'   Matrix shape: {kb_vectors.shape}  ({kb_vectors.shape[0]} questions × {kb_vectors.shape[1]} features)')

## 🎯 Step 4 — Intent Classifier (Naive Bayes)
Detects **what kind of question** the user is asking so the bot can frame its answer appropriately.

In [ ]:
# Intent detection using regex rules — no labelled data needed
INTENT_RULES = [
    ('explain',  r'\b(explain|why|how does|how do|reason|cause|because)\b'),
    ('compare',  r'\b(difference|compare|similar|versus|vs|between|contrast)\b'),
    ('define',   r'\b(what is|what are|define|meaning of|definition)\b'),
    ('evidence', r'\b(evidence|proof|show|indicate|data|experiment|result)\b'),
    ('predict',  r'\b(what happen|what will|predict|expect|would happen|if)\b'),
    ('describe', r'\b(describe|tell me about|explain what|what does it look)\b'),
]

INTENT_OPENERS = {
    'explain'  : 'Here is an explanation:',
    'compare'  : 'Here is a comparison:',
    'define'   : 'Here is the definition:',
    'evidence' : 'Based on the evidence:',
    'predict'  : 'Based on what we know:',
    'describe' : 'Here is a description:',
    'general'  : 'Here is what I found:',
}

def detect_intent(question: str) -> str:
    q = question.lower()
    for intent, pattern in INTENT_RULES:
        if re.search(pattern, q):
            return intent
    return 'general'

# Test
test_qs = [
    'What is the difference between a rock and a mineral?',
    'How does erosion happen?',
    'What is the evidence for more acid in OJ1?'
]
for q in test_qs:
    print(f'  "{q[:55]}" → intent: {detect_intent(q)}')

## 💬 Step 5 — Chatbot Engine
The core `ask()` function: retrieves best answer, detects intent, formats a natural response.

In [ ]:
CONFIDENCE_THRESHOLD = 0.15   # minimum similarity to give an answer

FALLBACK_RESPONSES = [
    "I don't have enough information to answer that confidently. Could you rephrase or ask something related to earth science, circuits, or plant biology?",
    "That question is outside my current knowledge base. Try asking about rocks, minerals, erosion, circuits, or sound.",
    "I'm not sure about that one. My knowledge covers topics like minerals, erosion, circuits, and plant science."
]

import random

def ask(user_question: str, top_k: int = 1, verbose: bool = True) -> dict:
    """
    Main chatbot function.
    
    Args:
        user_question : any question the user types
        top_k         : how many candidate answers to retrieve
        verbose       : print formatted response
    
    Returns dict with: answer, confidence, intent, matched_question
    """
    if not user_question.strip():
        return {'answer': 'Please type a question!', 'confidence': 0}

    # ── 1. Preprocess the user question ──
    q_clean  = preprocess(user_question)
    q_vector = retrieval_tfidf.transform([q_clean])

    # ── 2. Compute cosine similarity against all KB questions ──
    similarities = cosine_similarity(q_vector, kb_vectors).flatten()
    top_indices  = similarities.argsort()[::-1][:top_k]
    best_idx     = top_indices[0]
    best_score   = similarities[best_idx]

    # ── 3. Detect question intent ──
    intent = detect_intent(user_question)
    opener = INTENT_OPENERS.get(intent, INTENT_OPENERS['general'])

    # ── 4. Build response ──
    if best_score < CONFIDENCE_THRESHOLD:
        answer          = random.choice(FALLBACK_RESPONSES)
        matched_question = None
        formatted        = answer
    else:
        matched_question = kb.iloc[best_idx]['question']
        raw_answer       = kb.iloc[best_idx]['reference_answer']
        answer           = raw_answer
        formatted        = f"{opener}\n\n{raw_answer}"

    result = {
        'answer'           : answer,
        'confidence'       : round(float(best_score), 3),
        'intent'           : intent,
        'matched_question' : matched_question
    }

    if verbose:
        sep = '─' * 60
        print(f'\n{sep}')
        print(f'🧑 You    : {user_question}')
        print(f'{sep}')
        if matched_question and matched_question.lower() != user_question.lower():
            print(f'🔍 Matched: {matched_question[:80]}...' if len(matched_question)>80 else f'🔍 Matched: {matched_question}')
        print(f'🎯 Intent : {intent.upper()}  |  Confidence: {best_score:.1%}')
        print(f'{sep}')
        print(f'🤖 Bot    : {formatted}')
        print(f'{sep}')

    return result

print('✅ Chatbot engine ready!')

## 🧪 Step 6 — Test with Sample Questions

In [ ]:
# These can be paraphrased — the bot should still find the right answer

print('\n===== TEST CASES =====')

ask('What is the difference between a rock and a mineral?')

In [ ]:
ask('How does erosion affect earth materials?')

In [ ]:
ask('Why would a paperclip scratch a mineral if a penny already scratched it?')

In [ ]:
ask('What happens to sound volume when you pluck a rubber band harder?')

In [ ]:
# Out-of-scope question — should trigger fallback
ask('What is the capital of France?')

## 🗨️ Step 7 — Live Chat Interface
Run this cell and chat with the bot directly in the notebook.

In [ ]:
print('=' * 60)
print('  🤖  SMART SCIENCE CHATBOT  —  Type to ask anything!')
print('  Topics: rocks, minerals, erosion, circuits, sound,')
print('          plants, heat, pulleys, mixtures, temperature')
print('  Type  "topics"  to see all questions I can answer.')
print('  Type  "quit"    to exit.')
print('=' * 60)

while True:
    user_input = input('\n🧑 You: ').strip()

    if not user_input:
        continue

    if user_input.lower() == 'quit':
        print('\n🤖 Bot: Goodbye! Keep learning! 👋')
        break

    if user_input.lower() == 'topics':
        print('\n🤖 Bot: Here are all the topics I know about:')
        for i, q in enumerate(kb['question'].tolist(), 1):
            display = q[:85] + '...' if len(q) > 85 else q
            print(f'  {i:2}. {display}')
        continue

    ask(user_input)